# Mycetoma AI — Cloud Training Pipeline
Self-supervised pretraining and multi-task finetuning for histopathology diagnosis.

## 1. Hardware Check

In [ ]:
!nvidia-smi
import torch
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Kaggle Authentication
**Action Required:** Upload your `kaggle.json` API token when prompted.

In [ ]:
!pip install -q kaggle

from google.colab import files
import os

if not os.path.exists('/root/.kaggle/kaggle.json'):
    print('Please upload your kaggle.json file:')
    uploaded = files.upload()
    if 'kaggle.json' in uploaded:
        !mkdir -p ~/.kaggle
        !cp kaggle.json ~/.kaggle/
        !chmod 600 ~/.kaggle/kaggle.json
        print('\n✅ Kaggle configured')
    else:
        raise RuntimeError('kaggle.json upload failed. Re-run this cell.')
else:
    print('✅ Kaggle already configured')

## 3. Clone Repository & Install Dependencies

In [ ]:
import os
%cd /content
if os.path.exists('MycetomaAi'):
    !rm -rf MycetomaAi
!git clone https://github.com/yashnaiduu/MycetomaAi.git
%cd MycetomaAi
!pip install -q -r requirements.txt

os.environ['PYTHONPATH'] = os.getcwd() + ':' + os.environ.get('PYTHONPATH', '')
print('\n✅ Repository cloned and dependencies installed')

## 4. Download & Verify Data

In [ ]:
!kaggle datasets download -d supporoot/mycetoma-ai-pretraining-data --unzip -p data/

import os
pretrain_dir = 'data/pretrain'
if os.path.exists(pretrain_dir):
    for d in os.listdir(pretrain_dir):
        full = os.path.join(pretrain_dir, d)
        if os.path.isdir(full):
            count = sum(1 for _, _, f in os.walk(full) for _ in f)
            print(f'  {d}: {count} files')
    print('\n✅ Data downloaded')
else:
    print('❌ data/pretrain not found. Check Kaggle dataset structure.')

## 5. Verify Imports

In [ ]:
from src.data.dataset import MycetomaDataset, MultiDatasetWrapper, get_image_paths
from src.data.transforms import SimCLRTransform
from src.models.model import MycetomaAIModel
from src.training.ssl_pretrainer import SSLPreTrainer
from src.training.trainer import MultiTaskTrainer
print('✅ All imports OK')

## 6. Run SSL Pretraining (Config-Driven + Auto-Resume)

In [ ]:
import os, yaml, datetime

run_id = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
config = {
    'stage': 'pretrain',
    'pretrain_data_dir': 'data/pretrain',
    'epochs': 100,
    'batch_size': 16,
    'lr': 3e-4,
    'checkpoint_dir': 'checkpoints/ssl',
    'checkpoint_every_n_epochs': 10,
    'resume_from': 'auto',
    'seed': 42,
    'num_workers': 4,
    'precision': 'fp16',
    'grad_accum_steps': 1,
    'wandb': False,
    'run_id': run_id,
    'log_dir': 'logs'
}
os.makedirs('configs', exist_ok=True)
config_path = f'configs/pretrain_{run_id}.yaml'
with open(config_path, 'w') as f:
    yaml.safe_dump(config, f)
print(f'✅ Config written → {config_path}')

In [ ]:
import os
os.environ['PYTHONPATH'] = os.getcwd() + ':' + os.environ.get('PYTHONPATH', '')

!python scripts/train.py --config {config_path}

## 7. (Optional) Run Multi-Task Finetuning
Requires the Mycetoma tissue dataset in `data/finetune/MyData/`. 

In [ ]:
# Example finetune config (edit paths once MyData + labels are ready)
# run_id = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
# finetune_config = {
#     'stage': 'finetune',
#     'finetune_data_dir': 'data/finetune/MyData',
#     'checkpoint': 'checkpoints/ssl/ssl_encoder_ep100.pth',
#     'epochs': 50,
#     'batch_size': 16,
#     'lr': 3e-4,
#     'checkpoint_dir': 'checkpoints/multitask',
#     'checkpoint_every_n_epochs': 5,
#     'resume_from': 'auto',
#     'seed': 42,
#     'num_workers': 4,
#     'precision': 'fp16',
#     'grad_accum_steps': 1,
#     'early_stop_patience': 5,
#     'wandb': False,
#     'run_id': run_id,
#     'log_dir': 'logs'
# }
# os.makedirs('configs', exist_ok=True)
# finetune_config_path = f'configs/finetune_{run_id}.yaml'
# with open(finetune_config_path, 'w') as f:
#     yaml.safe_dump(finetune_config, f)
# !python scripts/train.py --config {finetune_config_path}

In [ ]:
# Example finetune config (edit paths once MyData + labels are ready)
# run_id = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
# finetune_config = {
#     'stage': 'finetune',
#     'finetune_data_dir': 'data/finetune/MyData',
#     'checkpoint': 'checkpoints/ssl/ssl_encoder_ep100.pth',
#     'epochs': 50,
#     'batch_size': 16,
#     'lr': 3e-4,
#     'checkpoint_dir': 'checkpoints/multitask',
#     'checkpoint_every_n_epochs': 5,
#     'resume_from': 'auto',
#     'seed': 42,
#     'num_workers': 4,
#     'precision': 'fp16',
#     'grad_accum_steps': 1,
#     'early_stop_patience': 5,
#     'wandb': False,
#     'run_id': run_id,
#     'log_dir': 'logs'
# }
# os.makedirs('configs', exist_ok=True)
# finetune_config_path = f'configs/finetune_{run_id}.yaml'
# with open(finetune_config_path, 'w') as f:
#     yaml.safe_dump(finetune_config, f)
# !python scripts/train.py --config {finetune_config_path}